# Bài 4: Phân lớp Iris bằng Neural Network và k-fold cross validation

**Họ tên:** Lê Nguyễn Anh Du  
**MSSV:** 6451071011

Thực hiện các yêu cầu a–g. Sử dụng 10 fold để đánh giá mô hình và chọn mô hình có F1-score trung bình cao nhất để dự đoán ba mẫu mới.

In [ ]:
print("Họ tên: Lê Nguyễn Anh Du | MSSV: 6451071011")
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from sklearn.datasets import load_iris
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

HO_TEN = "Lê Nguyễn Anh Du"
MSSV = "6451071011"
def in_thong_tin():
    print(f"Họ tên: {HO_TEN} | MSSV: {MSSV}")

tf.keras.utils.set_random_seed(42)
in_thong_tin()
print("Đã nhập các thư viện cần thiết và thiết lập seed.")

## a) Đọc bộ dữ liệu Iris từ sklearn

In [ ]:
in_thong_tin()
iris = load_iris()
X = iris.data
y = iris.target
print("Kích thước dữ liệu:", X.shape)
print("Tên các lớp:", iris.target_names)

## b) Chuẩn hóa dữ liệu về đoạn [0, 1]

Theo cách thực hiện của Bài 1, chia từng thuộc tính cho giá trị lớn nhất trong cột tương ứng. Giữ lại các giá trị lớn nhất này để chuẩn hóa ba mẫu mới ở phần g.

In [ ]:
in_thong_tin()
max_moi_cot = X.max(axis=0)
X_scaled = X / max_moi_cot
print("Giá trị lớn nhất mỗi thuộc tính:", max_moi_cot)
print("Khoảng giá trị sau chuẩn hóa:", X_scaled.min(), "đến", X_scaled.max())
print("5 mẫu sau chuẩn hóa:\n", X_scaled[:5])

## c) Định nghĩa hàm `create_model`

Mạng giống Bài 1.h: hai lớp ẩn Dense 10 và 20 neuron dùng ReLU, lớp đầu ra 3 neuron dùng Softmax. Biên dịch bằng Adam, sparse categorical crossentropy và accuracy.

In [ ]:
def create_model():
    model = Sequential([
        Input(shape=(X_scaled.shape[1],)),
        Dense(10, activation="relu"),
        Dense(20, activation="relu"),
        Dense(len(iris.target_names), activation="softmax")
    ])
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

in_thong_tin()
print("Đã định nghĩa mô hình với hai lớp ẩn và một lớp đầu ra.")

## d) Huấn luyện và đánh giá bằng 10-fold cross validation

Mỗi fold tạo một mô hình mới và huấn luyện 100 epoch với batch size 1 như Bài 1.h. Precision, recall và F1-score của fold được tính theo **macro average**, tức là trung bình đều trên ba lớp. Đồng thời lưu các chỉ số riêng của từng lớp để tổng hợp ở phần f.

In [ ]:
in_thong_tin()
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
class_labels = np.arange(len(iris.target_names))
models = []
accuracy_per_fold = []
precision_per_fold = []
recall_per_fold = []
f1_per_fold = []
class_precision_per_fold = []
class_recall_per_fold = []
class_f1_per_fold = []

for fold_number, (train_index, val_index) in enumerate(kfold.split(X_scaled), start=1):
    X_train_fold, X_val_fold = X_scaled[train_index], X_scaled[val_index]
    y_train_fold, y_val_fold = y[train_index], y[val_index]

    model = create_model()
    model.fit(X_train_fold, y_train_fold, epochs=100, batch_size=1, verbose=0)
    probabilities = model.predict(X_val_fold, verbose=0)
    y_pred_fold = np.argmax(probabilities, axis=1)

    accuracy = accuracy_score(y_val_fold, y_pred_fold)
    precision_by_class, recall_by_class, f1_by_class, _ = precision_recall_fscore_support(
        y_val_fold, y_pred_fold, labels=class_labels, average=None, zero_division=0
    )
    macro_precision = float(np.mean(precision_by_class))
    macro_recall = float(np.mean(recall_by_class))
    macro_f1 = float(np.mean(f1_by_class))

    models.append(model)
    accuracy_per_fold.append(float(accuracy))
    precision_per_fold.append(macro_precision)
    recall_per_fold.append(macro_recall)
    f1_per_fold.append(macro_f1)
    class_precision_per_fold.append(precision_by_class)
    class_recall_per_fold.append(recall_by_class)
    class_f1_per_fold.append(f1_by_class)

    in_thong_tin()
    print(f"Fold {fold_number:02d}/10 | Accuracy: {accuracy:.4f} | Precision: {macro_precision:.4f} | Recall: {macro_recall:.4f} | F1-score: {macro_f1:.4f}")

in_thong_tin()
print(f"Đã lưu {len(models)} mô hình sau 10 fold.")

## e) Chỉ số trung bình qua 10 fold

In [ ]:
in_thong_tin()
print(f"Average Accuracy: {np.mean(accuracy_per_fold):.4f}")
print(f"Average Recall: {np.mean(recall_per_fold):.4f}")
print(f"Average Precision: {np.mean(precision_per_fold):.4f}")
print(f"Average F1-score: {np.mean(f1_per_fold):.4f}")

## f) Recall, Precision và F1-score trung bình cho từng lớp

Giá trị của mỗi lớp là trung bình số học qua 10 fold. Khi một lớp không có mẫu thật hoặc dự đoán trong fold, chỉ số không xác định được gán bằng 0.

In [ ]:
in_thong_tin()
average_class_recall = np.mean(np.array(class_recall_per_fold), axis=0)
average_class_precision = np.mean(np.array(class_precision_per_fold), axis=0)
average_class_f1 = np.mean(np.array(class_f1_per_fold), axis=0)
class_summary = pd.DataFrame({
    "Lớp": iris.target_names,
    "Average Recall": average_class_recall,
    "Average Precision": average_class_precision,
    "Average F1-score": average_class_f1
})
display(class_summary.round(4))

## g) Dự đoán ba mẫu dữ liệu mới bằng mô hình có F1-score tốt nhất

Chọn fold có F1-score macro cao nhất trên phần validation. Chuẩn hóa mẫu mới bằng cùng giá trị lớn nhất ở phần b.

In [ ]:
in_thong_tin()
best_index = int(np.argmax(f1_per_fold))
best_model = models[best_index]
print(f"Mô hình tốt nhất: fold {best_index + 1}, F1-score macro = {f1_per_fold[best_index]:.4f}")

new_samples = np.array([
    [6.2, 2.9, 4.3, 1.3],
    [5.1, 3.5, 1.4, 0.2],
    [7.3, 2.8, 6.4, 2.1]
], dtype="float32")
new_samples_scaled = new_samples / max_moi_cot
new_probabilities = best_model.predict(new_samples_scaled, verbose=0)
new_predictions = np.argmax(new_probabilities, axis=1)
for sample_number, (sample, class_index) in enumerate(zip(new_samples, new_predictions), start=1):
    print(f"Mẫu {sample_number} {sample.tolist()} → {iris.target_names[class_index]} (nhãn {class_index})")